## TOKENIZACIÓN, LEMATIZACIÓN Y STOPWORDS

### 1. TOKENIZACIÓN 

Primero cogeremos el dataset traducido y lo tokenizaremos. Para esto, usaremos la librería `nltk` que es muy común en procesamiento de lenguaje natural. 

In [ ]:
import pandas as pd 
import spacy

df=pd.read_csv('datos/procesados/df_comentarios_traducidos.csv', encoding='utf-8-sig')
nlp=spacy.load('es_core_news_sm')

def tokenize(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return ""
    
    # Usamos nlp.tokenizer en lugar de nlp()
    doc = nlp.tokenizer(texto)
    
    # solo tokenizar
    tokens = [token.text for token in doc]
    # unir tokens en una cadena de texto
    return " ".join(tokens)


# APLICAMOS A LAS COLUMNAS CREANDO COLUMNAS NUEVAS
print("Procesando 'comentario_general'...")
df['general_tokenizado'] = df['comentario_general'].apply(tokenize)

print("Procesando 'positivo'...")
df['positivo_tokenizado'] = df['positivo'].apply(tokenize)

print("Procesando 'negativo'...")
df['negativo_tokenizado'] = df['negativo'].apply(tokenize)

print("¡Preprocesado completado!")
display(df[['comentario_general', 'general_tokenizado']].head())
display(df[['positivo', 'positivo_tokenizado']].head())
display(df[['negativo', 'negativo_tokenizado']].head())


Procesando 'comentario_general'...
Procesando 'positivo'...
Procesando 'negativo'...
¡Preprocesado completado!


,comentario_general,general_tokenizado
0,espectacular,espectacular
1,muy bien,muy bien
2,fantástico,fantástico
3,todo perfecto,todo perfecto
4,una nit del foc inolvidable.,una nit del foc inolvidable .


,positivo,positivo_tokenizado
0,espectacular todo! restaurante; servicio; habi...,espectacular todo ! restaurante ; servicio ; h...
1,NaN,
2,"habitaciones amplias, limpieza y ubicación","habitaciones amplias , limpieza y ubicación"
3,el trato en recepción,el trato en recepción
4,las vistas y el tamaño de la habitación.,las vistas y el tamaño de la habitación .


,negativo,negativo_tokenizado
0,todo me gustó,todo me gustó
1,NaN,
2,NaN,
3,el estacionamiento,el estacionamiento
4,"la calefacción central, sólo eso.","la calefacción central , sólo eso ."


Quitamos los caracteres especiales, los números y los signos de puntuación, y convertimos todo a minúsculas para normalizar el texto. 

In [6]:
import re

def limpiar_tokens(texto_tokenizado):
    if pd.isna(texto_tokenizado) or str(texto_tokenizado).strip() == "":
        return ""
    
    tokens = str(texto_tokenizado).split()
    tokens_limpios = []
    
    for token in tokens:
        # Condición 1: ¿Es 100% puntuación o símbolos raros?
        es_puntuacion = re.match(r'^[\W_]+$', token)
        
        # Condición 2: ¿Es 100% un número? (Detecta "402", "10", "2026")
        es_numero = token.isdigit()
        
        # Si NO es puntuación Y TAMPOCO es un número, lo guardamos
        if not es_puntuacion and not es_numero: 
             tokens_limpios.append(token)

    return " ".join(tokens_limpios)

# APLICAMOS A LAS COLUMNAS
print("Limpiando tokens en 'general_tokenizado'...")
df['general_tokenizado'] = df['general_tokenizado'].apply(limpiar_tokens)

print("Limpiando tokens en 'positivo_tokenizado'...")
df['positivo_tokenizado'] = df['positivo_tokenizado'].apply(limpiar_tokens)

print("Limpiando tokens en 'negativo_tokenizado'...")
df['negativo_tokenizado'] = df['negativo_tokenizado'].apply(limpiar_tokens)

print("¡Limpieza de puntuación y números completada!")

display(df[['comentario_general', 'general_tokenizado']].head())
display(df[['positivo', 'positivo_tokenizado']].head())
display(df[['negativo', 'negativo_tokenizado']].head())

Limpiando tokens en 'general_tokenizado'...
Limpiando tokens en 'positivo_tokenizado'...
Limpiando tokens en 'negativo_tokenizado'...
¡Limpieza de puntuación y números completada!


,comentario_general,general_tokenizado
0,espectacular,espectacular
1,muy bien,muy bien
2,fantástico,fantástico
3,todo perfecto,todo perfecto
4,una nit del foc inolvidable.,una nit del foc inolvidable


,positivo,positivo_tokenizado
0,espectacular todo! restaurante; servicio; habi...,espectacular todo restaurante servicio habitac...
1,NaN,
2,"habitaciones amplias, limpieza y ubicación",habitaciones amplias limpieza y ubicación
3,el trato en recepción,el trato en recepción
4,las vistas y el tamaño de la habitación.,las vistas y el tamaño de la habitación


,negativo,negativo_tokenizado
0,todo me gustó,todo me gustó
1,NaN,
2,NaN,
3,el estacionamiento,el estacionamiento
4,"la calefacción central, sólo eso.",la calefacción central sólo eso


### LEMATIZACIÓN 

La lematización es el proceso de reducir las palabras a su forma base o raíz (lema). Esto ayuda a agrupar diferentes formas de una palabra para que se analicen como una sola entidad. Por ejemplo, "correr", "corriendo" y "corrí" se reducirían a "correr".

In [ ]:
import spacy
import pandas as pd
import re

# 1. Cargamos el modelo avanzado de spaCy
nlp = spacy.load('es_core_news_sm')

# 2. Creamos la función optimizada con nlp.pipe
def lematizar_en_lotes(textos):
    textos_limpios = []
    
    # nlp.pipe procesa los textos en paralelo (batch_size=1000, n_process=-1 usa todos los núcleos)
    for doc in nlp.pipe(textos, batch_size=1000, n_process=-1):
        tokens_finales = []
        for token in doc:
            # Extraemos el lema de la palabra
            lema = token.lemma_
                
            # Filtro 1: ¿Es puntuación nativa de spaCy? (puntos, comas...)
            if token.is_punct:
                continue
                
            # Filtro 2: ¿Es un número o ruido puro? (Detecta "402", "10", o tokens como "!!!")
            es_ruido = re.match(r'^[\W_]+$', lema)
            es_numero = lema.isdigit()
            
            if not es_ruido and not es_numero:
                # Si pasa todos los filtros, guardamos el lema
                tokens_finales.append(lema)
                
        # Unimos los lemas de esta frase y los guardamos en la lista maestra
        textos_limpios.append(" ".join(tokens_finales))
        
    return textos_limpios

# 3. Preparamos los datos (nlp.pipe necesita listas de texto puro, no series de pandas con nulos)
textos_general = df['comentario_general'].fillna("").astype(str).tolist()
textos_positivo = df['positivo'].fillna("").astype(str).tolist()
textos_negativo = df['negativo'].fillna("").astype(str).tolist()

# 4. Aplicamos el procesamiento a máxima velocidad
print("⏳ Lematizando 'comentario_general' en lotes paralelos...")
df['general_lematizado'] = lematizar_en_lotes(textos_general)

print("⏳ Lematizando 'positivo' en lotes paralelos...")
df['positivo_lematizado'] = lematizar_en_lotes(textos_positivo)

print("⏳ Lematizando 'negativo' en lotes paralelos...")
df['negativo_lematizado'] = lematizar_en_lotes(textos_negativo)

print("✅ ¡Lematización y limpieza completada!")

# Comprobamos el resultado final
display(df[['comentario_general', 'general_tokenizado', 'general_lematizado']].head())
display(df[['positivo', 'positivo_tokenizado', 'positivo_lematizado']].head())
display(df[['negativo', 'negativo_tokenizado', 'negativo_lematizado']].head())

### STOPWORDS

Las stopwords son palabras muy comunes que no aportan mucho significado por sí solas, como "el", "la", "de", "que", etc. Estas palabras suelen eliminarse del texto para reducir el ruido y enfocarse en las palabras más relevantes. También se pueden añadir palabras específicas del proyecto que se consideren irrelevantes, como "hotel", "habitación", "valencia", etc.

In [7]:
import pandas as pd
from spacy.lang.es.stop_words import STOP_WORDS

# 1. Creamos un conjunto (set) con las stopwords de spaCy. 
# Usar un 'set' en Python hace que la búsqueda sea ultrarrápida.
stopwords_espanol = set(STOP_WORDS)

# 2. Añadimos palabras específicas del proyecto
#palabras_extra = {"hotel", "habitación", "habitacion", "valencia", "q", "d", "x"}
#stopwords_espanol.update(palabras_extra)

# 3. Función para filtrar palabras
def quitar_stopwords(texto):
    # Si está vacío o es nulo, devolvemos vacío
    if pd.isna(texto) or str(texto).strip() == "":
        return ""
    
    # Separamos el texto por espacios
    palabras = str(texto).split()
    
    # Nos quedamos solo con las palabras que NO están en la lista de stopwords
    palabras_filtradas = [palabra for palabra in palabras if palabra not in stopwords_espanol]
    
    # Volvemos a unir las palabras supervivientes
    return " ".join(palabras_filtradas)


# 4. APLICAMOS LA FUNCIÓN A LAS COLUMNAS TOKENIZADAS
print("Quitando stopwords a las columnas tokenizadas...")
df['general_tokenizado_sinstop'] = df['general_tokenizado'].apply(quitar_stopwords)
df['positivo_tokenizado_sinstop'] = df['positivo_tokenizado'].apply(quitar_stopwords)
df['negativo_tokenizado_sinstop'] = df['negativo_tokenizado'].apply(quitar_stopwords)

# 5. APLICAMOS LA FUNCIÓN A LAS COLUMNAS LEMATIZADAS
#print("Quitando stopwords a las columnas lematizadas...")
#df['general_lematizado_sinstop'] = df['general_lematizado'].apply(quitar_stopwords)
#df['positivo_lematizado_sinstop'] = df['positivo_lematizado'].apply(quitar_stopwords)
#df['negativo_lematizado_sinstop'] = df['negativo_lematizado'].apply(quitar_stopwords)

print("✅ ¡Stopwords fulminadas!")

# 6. Comprobamos (Antes vs Después)
print("\n--- EJEMPLO GENERAL TOKENIZADO ---")
display(df[['general_tokenizado', 'general_tokenizado_sinstop']].head())
display(df[['positivo_tokenizado', 'positivo_tokenizado_sinstop']].head())
display(df[['negativo_tokenizado', 'negativo_tokenizado_sinstop']].head())

#print("\n--- EJEMPLO GENERAL LEMATIZADO ---")
#display(df[['general_lematizado', 'general_lematizado_sinstop']].head())
#display(df[['positivo_lematizado', 'positivo_lematizado_sinstop']].head())
#display(df[['negativo_lematizado', 'negativo_lematizado_sinstop']].head())

Quitando stopwords a las columnas tokenizadas...
✅ ¡Stopwords fulminadas!

--- EJEMPLO GENERAL TOKENIZADO ---


,general_tokenizado,general_tokenizado_sinstop
0,espectacular,espectacular
1,muy bien,
2,fantástico,fantástico
3,todo perfecto,perfecto
4,una nit del foc inolvidable,nit foc inolvidable


,positivo_tokenizado,positivo_tokenizado_sinstop
0,espectacular todo restaurante servicio habitac...,espectacular restaurante servicio habitación t...
1,,
2,habitaciones amplias limpieza y ubicación,habitaciones amplias limpieza ubicación
3,el trato en recepción,trato recepción
4,las vistas y el tamaño de la habitación,vistas tamaño habitación


,negativo_tokenizado,negativo_tokenizado_sinstop
0,todo me gustó,gustó
1,,
2,,
3,el estacionamiento,estacionamiento
4,la calefacción central sólo eso,calefacción central


Guardar el resultado de la tokenización, lematización y eliminación de stopwords en nuevas columnas del DataFrame para mantener el original intacto. 

In [ ]:
#guardar csv 
df.to_csv('datos/procesados/df_comentarios_final.csv', index=False, encoding='utf-8-sig')